# How Can We See Neural-Weight Topology?

Original educational notebook accompanying a deep dive on arXiv:2611.13701. **Not** a reproduction of published Fashion-MNIST results. Run from the project root directory.

In [ ]:
from weight_topology_lab.topology import rips_h0_h1, summarize, normalize_rows, betti
import numpy as np
import matplotlib.pyplot as plt

## 1. A circle gives a persistent H1 loop

Each point can stand for one neuron incoming-weight row. Start with a controllable 2D point cloud.

In [ ]:
angles = np.linspace(0, 2*np.pi, 12, endpoint=False)
circle = np.column_stack([np.cos(angles), np.sin(angles)])
d = rips_h0_h1(circle)
print("H0:", summarize(d[0]))
print("H1:", summarize(d[1]))
plt.scatter(circle[:,0],circle[:,1]); plt.axis("equal"); plt.show()

## 2. Raw versus normalized neuron vectors

Unit normalization removes radial scale but not angular differences or function-equivalent reparameterizations.

In [ ]:
scaled = circle * np.linspace(0.3, 2.5, len(circle))[:,None]
for label, pts in (("raw",scaled),("row-normalized",normalize_rows(scaled))):
    h1 = rips_h0_h1(pts)[1]
    print(label, summarize(h1))

## 3. Track topology during toy supervised training

Three models share initialization and minibatch order, but receive correct, fixed-random, or changing-random labels. This is a *2D synthetic dataset* and uses only 12 first-layer neurons.

In [ ]:
from weight_topology_lab.experiment import training_run
runs = training_run(seed=42, epochs=12)
for name, rows in runs.items():
    print(name, "final true test accuracy:", round(rows[-1]["test_accuracy_on_true_labels"], 3), "H1 total:", round(rows[-1]["h1_total"],3))

In [ ]:
fig, axes=plt.subplots(1,2,figsize=(11,4))
for name, rows in runs.items():
    axes[0].plot([r["epoch"] for r in rows],[r["test_accuracy_on_true_labels"] for r in rows],label=name)
    axes[1].plot([r["epoch"] for r in rows],[r["h1_total"] for r in rows],label=name)
axes[0].set_title("True-label accuracy (toy)")
axes[1].set_title("First-layer normalized H1 (toy)")
for ax in axes: ax.set_xlabel("Epoch"); ax.legend(fontsize=8); ax.grid(alpha=.2)
plt.show()

## 4. Interpret with care

A visually distinct persistence trajectory is not evidence that topology adds predictive value beyond ordinary metrics. The original paper reports 60.0% early classification using topology, 66.7% with conventional diagnostics, and 60.0% combined. Its five paired seeds cannot reach a two-sided exact sign-flip p-value below 0.0625. If you run the optional Fashion-MNIST-scale experiment, analyze results at **seed level**, not by pretending each checkpoint or persistence interval is an independent replicate.